Importing the Required Packages 

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

Reading the datas 

In [2]:
books=pd.read_csv("books.csv")

In [3]:
book_tags=pd.read_csv("book_tags.csv")

In [4]:
tags=pd.read_csv("tags.csv")

Attaching tag names to each row of the book data 

In [5]:
book_tags = book_tags.merge(tags, on="tag_id", how="left")

In [6]:
print(book_tags) # to have a look 

        goodreads_book_id  tag_id   count           tag_name
0                       1   30574  167697            to-read
1                       1   11305   37174            fantasy
2                       1   11557   34173          favorites
3                       1    8717   12986  currently-reading
4                       1   33114   12716        young-adult
...                   ...     ...     ...                ...
999907           33288638   21303       7          neighbors
999908           33288638   17271       7    kindleunlimited
999909           33288638    1126       7       5-star-reads
999910           33288638   11478       7        fave-author
999911           33288638   27939       7           slowburn

[999912 rows x 4 columns]


In [7]:
top_tags=(book_tags.sort_values(['goodreads_book_id','count'],ascending=[True,False]).groupby('goodreads_book_id').head(5))

In [8]:
print(top_tags)

        goodreads_book_id  tag_id   count           tag_name
0                       1   30574  167697            to-read
1                       1   11305   37174            fantasy
2                       1   11557   34173          favorites
3                       1    8717   12986  currently-reading
4                       1   33114   12716        young-adult
...                   ...     ...     ...                ...
999812           33288638   30574   14116            to-read
999813           33288638    8717    1196  currently-reading
999814           33288638   26138     385            romance
999815           33288638   11557     354          favorites
999816           33288638    8055     216       contemporary

[50000 rows x 4 columns]


Explaining the merge function:
merging top_tags with books dataset on the basis of the goodreads_book_id column on the Right( left shld be written).

In [9]:
books_tags_joined=books.merge(top_tags,on="goodreads_book_id",how="left")

In [10]:
print(books_tags_joined)

       book_id  goodreads_book_id  best_book_id  work_id  books_count  \
0            1            2767052       2767052  2792775          272   
1            1            2767052       2767052  2792775          272   
2            1            2767052       2767052  2792775          272   
3            1            2767052       2767052  2792775          272   
4            1            2767052       2767052  2792775          272   
...        ...                ...           ...      ...          ...   
49995    10000               8914          8914    11817           31   
49996    10000               8914          8914    11817           31   
49997    10000               8914          8914    11817           31   
49998    10000               8914          8914    11817           31   
49999    10000               8914          8914    11817           31   

            isbn        isbn13          authors  original_publication_year  \
0      439023483  9.780439e+12  Suzanne Colli

COMBINING ALL TAGS OF THE BOOK INTO A SINGLE STRING

books_tags_joined.groupby(['book_id','title','authors']) Groups all rows by book_id, title, and authors. So for each book, you collect all its tags together.
['tag_name'] From each group, we only care about the tag_name column (the actual text labels of tags like “fantasy”, “romance”, etc.).
.apply(lambda x: " ".join(x.dropna())) For each group (one book), take the column of tag names (x is a Series of tag names).

x.dropna() removes any missing values.

" ".join(...) combines all the tags into a single string separated by spaces. Example: ["fantasy", "magic", "adventure"] → "fantasy magic adventure" This way, each book ends up with one “document” of its tags.
.reset_index()

After grouping, the grouping keys (book_id, title, authors) become the index.

reset_index() turns them back into normal columns.

In [11]:
books_with_tags=books_tags_joined.groupby(["book_id","title","authors"])["tag_name"].apply(lambda x: " ".join(x.dropna())).reset_index()

In [12]:
print(books_with_tags)

      book_id                                              title  \
0           1            The Hunger Games (The Hunger Games, #1)   
1           2  Harry Potter and the Sorcerer's Stone (Harry P...   
2           3                            Twilight (Twilight, #1)   
3           4                              To Kill a Mockingbird   
4           5                                   The Great Gatsby   
...       ...                                                ...   
9995     9996                          Bayou Moon (The Edge, #2)   
9996     9997  Means of Ascent (The Years of Lyndon Johnson, #2)   
9997     9998                              The Mauritius Command   
9998     9999  Cinderella Ate My Daughter: Dispatches from th...   
9999    10000                                The First World War   

                          authors  \
0                 Suzanne Collins   
1     J.K. Rowling, Mary GrandPré   
2                 Stephenie Meyer   
3                      Harper Lee  

Building A single text field for each book ie combining the title authors and the tagnames inside a new column named features

In [13]:
books_with_tags["features"]=(books_with_tags["title"]+" "+books_with_tags["authors"]+" "+books_with_tags["tag_name"])

In [14]:
print(books_with_tags["features"])

0       The Hunger Games (The Hunger Games, #1) Suzann...
1       Harry Potter and the Sorcerer's Stone (Harry P...
2       Twilight (Twilight, #1) Stephenie Meyer young-...
3       To Kill a Mockingbird Harper Lee classics favo...
4       The Great Gatsby F. Scott Fitzgerald classics ...
                              ...                        
9995    Bayou Moon (The Edge, #2) Ilona Andrews to-rea...
9996    Means of Ascent (The Years of Lyndon Johnson, ...
9997    The Mauritius Command Patrick O'Brian to-read ...
9998    Cinderella Ate My Daughter: Dispatches from th...
9999    The First World War John Keegan to-read histor...
Name: features, Length: 10000, dtype: object


In [15]:
tfidf=TfidfVectorizer(stop_words="english")

In [16]:
tfidf_matrix=tfidf.fit_transform(books_with_tags['features'])

In [17]:
cosine_sim=cosine_similarity(tfidf_matrix,tfidf_matrix)

In [18]:
indices=pd.Series(books_with_tags.index,books_with_tags["title"]).drop_duplicates()

In [19]:
print(indices)

title
The Hunger Games (The Hunger Games, #1)                                                         0
Harry Potter and the Sorcerer's Stone (Harry Potter, #1)                                        1
Twilight (Twilight, #1)                                                                         2
To Kill a Mockingbird                                                                           3
The Great Gatsby                                                                                4
                                                                                             ... 
Bayou Moon (The Edge, #2)                                                                    9995
Means of Ascent (The Years of Lyndon Johnson, #2)                                            9996
The Mauritius Command                                                                        9997
Cinderella Ate My Daughter: Dispatches from the Frontlines of the New Girlie-Girl Culture    9998
The First Worl

Writing the Recommendation Function 

In [20]:
def recommend_books(title,n=5):
    if title not in indices:
        return"Book not found in Dataset"
    idx=indices[title]
    sim_scores=list(enumerate(cosine_sim[idx]))
    sim_scores=sorted(sim_scores,key=lambda x:x[1],reverse=True)
    sim_scores=sim_scores[1:n+1]
    book_indices=[i[0] for i in sim_scores]
    return books_with_tags[["title","authors"]].iloc[book_indices]

In [30]:
title=input("Enter the name of the book for which you want Recommendations: " )
n=int(input("Enter the number of recommendations you want: "))
recommend_books(title,n)

Enter the name of the book for which you want Recommendations:  The Fault in Our Stars
Enter the number of recommendations you want:  5


,title,authors
87,Paper Towns,John Green
73,Looking for Alaska,John Green
274,An Abundance of Katherines,John Green
9537,Agent to the Stars,John Scalzi
381,"Will Grayson, Will Grayson","John Green, David Levithan"


In [31]:
recommend_books("Harry Potter and the Goblet of Fire (Harry Potter, #4)",7)

,title,authors
2100,"The Harry Potter Collection 1-4 (Harry Potter,...","J.K. Rowling, Mary GrandPré"
3274,"Harry Potter Boxed Set, Books 1-5 (Harry Potte...","J.K. Rowling, Mary GrandPré"
24,Harry Potter and the Deathly Hallows (Harry Po...,"J.K. Rowling, Mary GrandPré"
22,Harry Potter and the Chamber of Secrets (Harry...,"J.K. Rowling, Mary GrandPré"
1,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré"
26,Harry Potter and the Half-Blood Prince (Harry ...,"J.K. Rowling, Mary GrandPré"
6140,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling
